In [ ]:
import pandas as pd

from src.data.loaders import (load_ml20m, split_and_reindex)

In [ ]:
config = {
    "ml-20m": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": "/kaggle/working/Actions-Speak-Louder-than-Words-research/ml-20m.zip", # https://grouplens.org/datasets/movielens/20m/
        "col_mapping": {
            "userid": "user_id",
            "movieid": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
}

In [ ]:
ml_df = load_ml20m(config["ml-20m"]["interactions_path"], config=config["ml-20m"])

In [ ]:
ml_train, ml_test, ml_data_description = split_and_reindex(ml_df, config=config["ml-20m"])

In [ ]:
datasets = {
    "ml-20m": {
        "train": ml_train,
        "test": ml_test,
        "desc": ml_data_description,
    },
}

In [20]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,ml-20m,18000184,353348,18353532,0.981,0.019,125040,4958,17951


## HSTU MovieLens experiment

Запуск HSTU-пайплайна на MovieLens

In [21]:
import torch

from src.training.hstu import (
    HSTUExperimentConfig,
    build_hstu_dataloaders,
    evaluate_hstu,
    train_hstu,
)
from src.models.hstu import HSTUModel


In [ ]:
hstu_config = HSTUExperimentConfig(
    max_seq_len=config["ml-20m"]["max_seq_len"],
    train_batch_size=128,
    eval_batch_size=128,
    num_epochs=10,
    learning_rate=1e-3,
    weight_decay=0.0,
    topk=100,
    embedding_dim=128,
    num_blocks=4,
    num_heads=4,
    linear_dim=64,
    attention_dim=64,
    input_dropout_rate=0.2,
    linear_dropout_rate=0.2,
    num_negatives=128,
    sampling_strategy="local",
    softmax_temperature=0.05,
    user_embedding_norm="l2_norm",
    enable_relative_attention_bias = True,
    l2_norm_embeddings=True,
    l2_norm_eps=1e-6,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

hstu_train_loader, hstu_eval_loader, hstu_train_dataset, hstu_eval_dataset, hstu_targets = build_hstu_dataloaders(
    train=ml_train,
    test=ml_test,
    max_seq_len=hstu_config.max_seq_len,
    train_batch_size=hstu_config.train_batch_size,
    eval_batch_size=hstu_config.eval_batch_size,
    user_col=ml_data_description["users"],
    item_col=ml_data_description["items"],
    time_col=ml_data_description["timestamp"],
    num_workers=hstu_config.num_workers,
)


len(hstu_train_dataset), len(hstu_eval_dataset), len(hstu_targets)


(170482, 4958, 4958)

In [23]:
hstu_model = HSTUModel(
    num_items=ml_data_description["n_items"],
    embedding_dim=hstu_config.embedding_dim,
    max_seq_len=hstu_config.max_seq_len,
    num_blocks=hstu_config.num_blocks,
    num_heads=hstu_config.num_heads,
    linear_dim=hstu_config.linear_dim,
    attention_dim=hstu_config.attention_dim,
    num_negatives=hstu_config.num_negatives,
    softmax_temperature=hstu_config.softmax_temperature,
    sampling_strategy=hstu_config.sampling_strategy,
    user_embedding_norm=hstu_config.user_embedding_norm,
    l2_norm_embeddings=hstu_config.l2_norm_embeddings,
    l2_norm_eps=hstu_config.l2_norm_eps,
    item_id_offset=hstu_config.item_id_offset,
    input_dropout_rate=hstu_config.input_dropout_rate,
    linear_dropout_rate=hstu_config.linear_dropout_rate,
    attn_dropout_rate=hstu_config.attn_dropout_rate,
)

hstu_model.to(hstu_config.device)
hstu_optimizer = torch.optim.AdamW(
    hstu_model.parameters(),
    lr=hstu_config.learning_rate,
    weight_decay=hstu_config.weight_decay,
)


In [24]:
params_sum = 0
trainable_sum = 0

for name, module in hstu_model.encoder.named_children():
    params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"{name}: total={params:,}, trainable={trainable:,}")

    params_sum += params
    trainable_sum += trainable
print()
print(f"params_sum={params_sum:,}, trainable_sum={trainable_sum:,}")

item_embeddings: total=2,297,856, trainable=2,297,856
pos_embeddings: total=25,600, trainable=25,600
input_dropout: total=0, trainable=0
blocks: total=661,056, trainable=661,056

params_sum=2,984,512, trainable_sum=2,984,512


In [25]:
hstu_losses, eval_history = train_hstu(
    model=hstu_model,
    train_loader=hstu_train_loader,
    optimizer=hstu_optimizer,
    num_epochs=hstu_config.num_epochs,
    device=hstu_config.device,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=ml_data_description["n_items"],
    topk=hstu_config.topk,
    filter_seen=hstu_config.filter_seen,
)
hstu_losses

epoch 1/10:   0%|          | 0/1331 [00:00<?, ?it/s]

HSTU evaluation:   0%|          | 0/39 [00:00<?, ?it/s]

epoch 1/10: loss=1.8239, coverage=0.3422, hitrate=0.8651, ndcg=0.1701, recall=0.2157


epoch 2/10:   0%|          | 0/1331 [00:00<?, ?it/s]

HSTU evaluation:   0%|          | 0/39 [00:00<?, ?it/s]

...

epoch 9/10: loss=1.2440, coverage=0.4923, hitrate=0.8756, ndcg=0.1834, recall=0.2299


epoch 10/10:   0%|          | 0/1331 [00:00<?, ?it/s]

HSTU evaluation:   0%|          | 0/39 [00:00<?, ?it/s]

epoch 10/10: loss=1.2373, coverage=0.5198, hitrate=0.8719, ndcg=0.1790, recall=0.2250


[1.8238878528665188,
 1.4271732214068578,
 1.3569728644784458,
 1.3180250297534923,
 1.292176107729762,
 1.2750715441850682,
 1.2620585739478234,
 1.2525655140471943,
 1.2439932819598605,
 1.2372645160293865]

In [26]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=ml_data_description["n_items"],
    topk=10,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/39 [00:00<?, ?it/s]

{'hitrate': 0.6113352158128278,
 'recall': 0.1748017473763768,
 'ndcg': 0.18158730311695181,
 'coverage': 0.20901342543590887}

In [27]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=ml_data_description["n_items"],
    topk=50,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/39 [00:00<?, ?it/s]

{'hitrate': 0.8154497781363453,
 'recall': 0.17900386556282316,
 'ndcg': 0.16479374997529003,
 'coverage': 0.4013146899894156}

In [28]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=ml_data_description["n_items"],
    topk=100,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/39 [00:00<?, ?it/s]

{'hitrate': 0.871924162968939,
 'recall': 0.22502361806148805,
 'ndcg': 0.1790439242245475,
 'coverage': 0.5198039106456465}

In [29]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=ml_data_description["n_items"],
    topk=200,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/39 [00:00<?, ?it/s]

{'hitrate': 0.9142799515933844,
 'recall': 0.3142921394080758,
 'ndcg': 0.21353366392262021,
 'coverage': 0.6635284942343045}

In [ ]:
checkpoint = {
    "epoch": hstu_config.num_epochs,
    "model_state_dict": hstu_model.state_dict(),
    "optimizer_state_dict": hstu_optimizer.state_dict(),
    "loss": hstu_losses,
}

torch.save(checkpoint, "ml-20m_checkpoint.pt")